# 循环神经网络基础

## 1. 序列模型

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import collections
import re

### 统计工具

处理序列数据时，我们需要统计工具来描述序列中的依赖关系。设 $x_t$ 为时间步 $t$ 的观测值，联合概率可分解为条件概率的乘积：

$$P(x_1, \ldots, x_T) = \prod_{t=1}^T P(x_t \mid x_{t-1}, \ldots, x_1)$$

### 自回归模型

自回归模型使用过去 $\tau$ 个观测值来预测当前值：

$$x_t = f(x_{t-1}, x_{t-2}, \ldots, x_{t-\tau}) + \epsilon_t$$

其中 $f$ 可以是线性模型或神经网络，$\epsilon_t$ 为噪声。

### 马尔可夫模型

马尔可夫假设认为当前状态只与最近 $\tau$ 个状态有关。一阶马尔可夫模型满足：

$$P(x_t \mid x_{t-1}, \ldots, x_1) = P(x_t \mid x_{t-1})$$

### 生成合成时间序列数据

In [ ]:
def generate_synthetic_data(num_steps=1000):
    x = torch.linspace(0, 20 * math.pi, num_steps)
    signal = torch.sin(x)
    noise = torch.normal(0, 0.1, size=(num_steps,))
    return signal + noise

data = generate_synthetic_data()
print('数据形状:', data.shape)
print('前10个值:', data[:10])

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(data.numpy(), label='含噪声的正弦波')
plt.xlabel('时间步')
plt.ylabel('值')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 训练MLP进行预测

使用过去 $\tau = 4$ 个时间步的值来预测当前值。

In [ ]:
tau = 4
features = torch.zeros((len(data) - tau, tau))
for i in range(tau):
    features[:, i] = data[i: len(data) - tau + i]
labels = data[tau:]

train_size = int(0.8 * len(features))
train_features, train_labels = features[:train_size], labels[:train_size]
test_features, test_labels = features[train_size:], labels[train_size:]
print('训练集大小:', train_features.shape)
print('测试集大小:', test_features.shape)

In [ ]:
model = nn.Sequential(
    nn.Linear(tau, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
train_losses = []
for epoch in range(num_epochs):
    model.train()
    pred = model(train_features).squeeze()
    loss = criterion(pred, train_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.6f}')

In [ ]:
model.eval()
with torch.no_grad():
    train_pred = model(train_features).squeeze()
    test_pred = model(test_features).squeeze()
    train_mse = criterion(train_pred, train_labels)
    test_mse = criterion(test_pred, test_labels)

print(f'训练 MSE: {train_mse:.6f}')
print(f'测试 MSE: {test_mse:.6f}')

plt.figure(figsize=(12, 4))
plt.plot(train_losses, label='训练损失')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(range(len(data)), data.numpy(), label='原始数据', alpha=0.5)
plt.plot(range(tau, train_size + tau), train_pred.numpy(), label='训练集预测', alpha=0.8)
plt.plot(range(train_size + tau, len(data)), test_pred.numpy(), label='测试集预测', alpha=0.8)
plt.axvline(x=train_size + tau, color='r', linestyle='--', alpha=0.5, label='训练/测试分界')
plt.xlabel('时间步')
plt.ylabel('值')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 文本预处理

文本预处理是将原始文本转换为模型可处理的数值形式的关键步骤。典型流程包括：读取文本、分词、构建词表、转换为序列。

In [ ]:
def read_time_machine():
    with open('time_machine.txt', 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

lines = read_time_machine()
print('总行数:', len(lines))
print('前3行:', lines[:3])

In [ ]:
def tokenize(lines, token='word'):
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        raise ValueError(f'未知分词方式: {token}')

word_tokens = tokenize(lines, 'word')
char_tokens = tokenize(lines, 'char')
print('词级别分词示例:', word_tokens[0][:10])
print('字符级别分词示例:', char_tokens[0][:20])

In [ ]:
class Vocab:
    def __init__(self, tokens, min_freq=0, reserved_tokens=None):
        if reserved_tokens is None:
            reserved_tokens = []
        counter = collections.Counter(t for line in tokens for t in line)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.unk = 0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self.token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

vocab = Vocab(word_tokens, min_freq=5)
print('词表大小:', len(vocab))
print('最常见词:', vocab.token_freqs[:10])

In [ ]:
def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
print('语料库大小:', len(corpus))
print('字符集大小:', len(vocab))
print('前20个索引:', corpus[:20])

In [ ]:
# 将索引序列转换回文本
print('前20个字符:', ''.join(vocab.to_tokens(corpus[:20])))

## 3. 语言模型

语言模型的目标是估计文本序列的联合概率 $P(x_1, x_2, \ldots, x_T)$。

In [ ]:
# 一元语法频率统计
unigram_freqs = [freq for token, freq in vocab.token_freqs]
print('一元语法最高频词:', vocab.token_freqs[:5])
print('一元语法样本数:', len(unigram_freqs))

### N元语法

N元语法基于 $(n-1)$ 阶马尔可夫假设，将条件概率近似为：

$$P(x_t \mid x_{t-1}, \ldots, x_1) \approx P(x_t \mid x_{t-1}, \ldots, x_{t-n+1})$$

常见选择：一元语法 (unigram)、二元语法 (bigram)、三元语法 (trigram)。

In [ ]:
# 统计二元语法和三元语法频率
def count_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(' '.join(tokens[i: i + n]))
    return collections.Counter(ngrams)

# 使用 word_tokens 进行统计
flat_tokens = [token for line in word_tokens for token in line]
bigram_counts = count_ngrams(flat_tokens, 2)
trigram_counts = count_ngrams(flat_tokens, 3)
print('二元语法示例:', list(bigram_counts.items())[:5])
print('三元语法示例:', list(trigram_counts.items())[:5])

In [ ]:
# 可视化 n-gram 频率分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (counts, name) in enumerate(zip(
    [unigram_freqs, list(bigram_counts.values()), list(trigram_counts.values())],
    ['一元语法', '二元语法', '三元语法']
)):
    axes[i].plot(counts[:50])
    axes[i].set_title(name)
    axes[i].set_xlabel('排名')
    axes[i].set_ylabel('频率')
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 困惑度 (Perplexity)

困惑度是评估语言模型的常用指标，定义为：

$$\text{Perplexity} = \exp\left(-\frac{1}{T}\sum_{t=1}^T \log P(x_t \mid x_{t-1}, \ldots, x_1)\right)$$

困惑度越低，模型越好。完美模型的困惑度为1。

In [ ]:
def compute_perplexity(probabilities):
    return torch.exp(-torch.mean(torch.log(probabilities)))

# 示例：随机概率下的困惑度
random_probs = torch.rand(100)
random_probs = random_probs / random_probs.sum()
print('随机概率下的困惑度:', compute_perplexity(random_probs).item())

## 4. 循环神经网络

### RNN概念

循环神经网络 (RNN) 通过引入隐藏状态来捕捉序列中的时间依赖关系。与传统的全连接网络不同，RNN的隐藏状态可以在时间步之间传递信息。

### 隐藏状态

在时间步 $t$，RNN的隐藏状态 $\mathbf{H}_t$ 同时依赖于当前输入 $\mathbf{X}_t$ 和上一时刻的隐藏状态 $\mathbf{H}_{t-1}$：

$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h)$$

输出为：

$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

其中 $\phi$ 是激活函数（通常为 $\tanh$）。

In [ ]:
# 基本RNN计算示例
batch_size, num_inputs, num_hiddens, num_outputs = 2, 4, 8, 4
X = torch.normal(0, 1, (batch_size, num_inputs))
W_xh = torch.normal(0, 1, (num_inputs, num_hiddens))
W_hh = torch.normal(0, 1, (num_hiddens, num_hiddens))
b_h = torch.zeros(num_hiddens,)
W_hq = torch.normal(0, 1, (num_hiddens, num_outputs))
b_q = torch.zeros(num_outputs,)

H = torch.zeros(batch_size, num_hiddens)
H = torch.tanh(X @ W_xh + H @ W_hh + b_h)
O = H @ W_hq + b_q
print('隐藏状态形状:', H.shape)
print('输出形状:', O.shape)

In [ ]:
# 多时间步RNN前向传播
num_steps = 3
X_seq = torch.normal(0, 1, (num_steps, batch_size, num_inputs))

H = torch.zeros(batch_size, num_hiddens)
outputs = []
for t in range(num_steps):
    H = torch.tanh(X_seq[t] @ W_xh + H @ W_hh + b_h)
    O = H @ W_hq + b_q
    outputs.append(O)

outputs = torch.stack(outputs)
print('RNN输出形状:', outputs.shape)
print('最终隐藏状态形状:', H.shape)

### RNN的梯度问题

RNN训练中面临两个主要的梯度问题：

1. **梯度消失**：当序列较长时，梯度在反向传播中指数级衰减，导致网络无法学习长期依赖。
2. **梯度爆炸**：梯度在反向传播中指数级增长，导致数值不稳定和训练发散。

解决这些问题的方法包括梯度裁剪 (gradient clipping)、门控机制 (GRU/LSTM) 等。

In [ ]:
# 梯度裁剪示例
def grad_clipping(net, theta):
    params = [p for p in net.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

# 简单的RNN层实现
class RNNLayer(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_xh = nn.Parameter(torch.normal(0, 0.01, (input_size, hidden_size)))
        self.W_hh = nn.Parameter(torch.normal(0, 0.01, (hidden_size, hidden_size)))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        self.W_hq = nn.Parameter(torch.normal(0, 0.01, (hidden_size, output_size)))
        self.b_q = nn.Parameter(torch.zeros(output_size))

    def forward(self, X, state=None):
        batch_size = X.shape[1]
        if state is None:
            state = torch.zeros(batch_size, self.hidden_size)
        H = torch.tanh(X @ self.W_xh + state @ self.W_hh + self.b_h)
        O = H @ self.W_hq + self.b_q
        return O, H

rnn_layer = RNNLayer(num_inputs, num_hiddens, num_outputs)
out, state = rnn_layer(X_seq)
print('RNN层输出形状:', out.shape)
print('隐藏状态形状:', state.shape)

In [ ]:
# 演示梯度爆炸和梯度裁剪
small_grad = torch.tensor([0.1])
large_grad = torch.tensor([50.0])
print('正常梯度经过tanh后:', torch.tanh(small_grad * 10).item())
print('经过多次相乘后梯度消失: 0.1^10 =', 0.1 ** 10)
print('经过多次相乘后梯度爆炸: 2^10 =', 2 ** 10)

theta = 1.0
print(f'\n裁剪阈值: {theta}')
print(f'裁剪前梯度范数: {large_grad.item():.2f}')
clipped = large_grad * theta / large_grad
print(f'裁剪后梯度范数: {clipped.item():.2f}')